# Three worlds where the obvious regression is wrong

Each of the three simulated worlds below has a known true effect, twenty thousand rows, and
no missing data. In each of them, the regression a competent analyst writes first returns a
confident, precise, wrong answer. What changes between them is *why* — a confounder, a
feedback loop into the treatment, a latent that no covariate can reach — and therefore which
estimator is licensed.

The graph names the route; these are the estimators for each. All are pure numpy with
classical standard errors, return a `LinearEstimate` spec, and their intervals are labelled
`wald` — a frequentist CI is a different object from a credible interval and the type says so.

In [ ]:
from axiom.identify import (
    CausalGraph, EndogeneityTest, FrontDoorRoute, InstrumentRoute, LinearEstimate,
    conditional_instruments, durbin_wu_hausman, frontdoor_admissible, frontdoor_linear,
    frontdoor_sets, hausman_iv_vs_ols, identify, instrument_admissible, instruments, ols,
    two_stage_least_squares, weak_instrument_check,
)
from axiom.sim import confounded_world, frontdoor_world, iv_world

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals

enable();  # every axiom result renders itself from here on

## Front-door criterion

In [ ]:
from axiom.display import show

fd_graph = CausalGraph.from_edges("X -> M, M -> Y, X <-> Y")
print(frontdoor_admissible(fd_graph, "X", "Y", ["M"]), frontdoor_sets(fd_graph, "X", "Y"))
route = FrontDoorRoute(mediators=("M",), treatment="X", outcome="Y")
show(route)
# a latent between X and M breaks condition (ii)
print(frontdoor_admissible(CausalGraph.from_edges("X -> M, M -> Y, X <-> Y, X <-> M"), "X", "Y", ["M"]))

## Instruments

`instruments` lists unconditional instruments (relevance in $G$, exclusion in $G_{\underline{X}}$);
`conditional_instruments` lists $(Z, W)$ pairs where the exclusion holds only given $W$.

In [ ]:
iv_graph = CausalGraph.from_edges("Z -> X, X -> Y, X <-> Y")
print(instruments(iv_graph, "X", "Y"), instrument_admissible(iv_graph, "X", "Y", "Z"))
print(instruments(CausalGraph.from_edges("Z -> X, Z -> Y, X -> Y, X <-> Y"), "X", "Y"), "<- exclusion violated")

civ = CausalGraph.from_edges("W -> Z, W -> Y, Z -> X, X -> Y, X <-> Y")
print(instruments(civ, "X", "Y"), conditional_instruments(civ, "X", "Y"))
print(InstrumentRoute(instrument="Z", conditioning=("W",), treatment="X", outcome="Y"))

## Estimating along the licensed route

Each `sim` world carries its truth. The naive estimate is biased by construction; the
estimator matching the verdict's route recovers the truth within its standard error.

In [ ]:
results = []

world = confounded_world()
frame = world.observed(world.simulate(20_000, seed=0))
truth = world.total_effect("X", "Y")
v = identify(world.graph, "X", "Y")

naive: LinearEstimate = ols(frame, "Y", "X")
adjusted = ols(frame, "Y", "X", covariates=v.adjustment_set)
print(f"truth {truth} | naive {naive.estimate:.3f} ± {naive.se:.3f} | adjusted {adjusted.estimate:.3f} ± {adjusted.se:.3f}")
print(adjusted.ci(0.95), adjusted.method, adjusted.covariates)
results += [("confounded · ols", naive, truth), (f"confounded · adjusted for {v.adjustment_set}", adjusted, truth)]

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
v = identify(world.graph, "X", "Y")
iv = two_stage_least_squares(frame, "Y", "X", instruments=[v.instrument])
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | 2sls {iv.estimate:.3f} ± {iv.se:.3f}")
print(iv.detail)
strength = weak_instrument_check(iv)
print(strength.name, strength.state, strength.statement)
results += [("instrumented · ols", ols(frame, "Y", "X"), world.total_effect("X", "Y")),
            ("instrumented · 2sls", iv, world.total_effect("X", "Y"))]

In [ ]:
world = frontdoor_world()
frame = world.observed(world.simulate(20_000, seed=2))
v = identify(world.graph, "X", "Y")
fd = frontdoor_linear(frame, "Y", "X", mediators=v.mediators)
print(f"truth {world.total_effect('X', 'Y')} | ols {ols(frame, 'Y', 'X').estimate:.3f} | front-door {fd.estimate:.3f} ± {fd.se:.3f}")
print(fd.ci(0.9))
results += [("front-door world · ols", ols(frame, "Y", "X"), world.total_effect("X", "Y")),
            ("front-door world · front-door", fd, world.total_effect("X", "Y"))]

In [ ]:
fig = intervals(
    [(label, est.estimate - tr, est.estimate - tr - 2 * est.se, est.estimate - tr + 2 * est.se)
     for label, est, tr in results],
    ref=0.0, ref_label="the truth",
    title="Six estimates, three worlds, one number each is trying to hit",
    subtitle="estimate minus the known truth, ±2 classical standard errors",
    x_title="error",
)
caption(fig, "Every second row sits on the line; every first row misses it by many standard "
             "errors. The estimator did not change between the pairs — the question of which "
             "one the graph licenses did.")

## Is the IV route needed? Endogeneity tests

An instrument is not free: 2SLS uses only the variation in the treatment that the instrument
explains, so its standard error is larger — twice OLS's on this world. Running it when
ordinary least squares was already unbiased is a real cost, and these tests are how you find
out whether you are paying it for nothing.

`durbin_wu_hausman` is the control-function form; `hausman_iv_vs_ols` contrasts the two
estimates directly. A degenerate contrast (negative variance difference) returns a typed
`Unverified`, not a fabricated p-value.

In [ ]:
world = iv_world()
frame = world.observed(world.simulate(20_000, seed=1))
dwh = durbin_wu_hausman(frame, "Y", "X", instruments=["Z"])
print(dwh if not isinstance(dwh, EndogeneityTest) else (dwh.conclusion, round(dwh.statistic, 2), dwh.p_value))

h = hausman_iv_vs_ols(ols(frame, "Y", "X"), two_stage_least_squares(frame, "Y", "X", instruments=["Z"]))
print(h if not isinstance(h, EndogeneityTest) else (h.conclusion, round(h.statistic, 2)))

## Guard rails

The estimators refuse the outcome on the right-hand side, missing columns, and too few rows —
loudly, never with a confidently wrong number. Regressing an outcome on itself is a typo that
produces an R² of 1.0 and a slide that nobody questions.

In [ ]:
refused = []
for label, bad in (
    ("outcome used as its own covariate", lambda: ols(frame, "Y", "X", covariates=["Y"])),
    ("a column that is not there", lambda: ols(frame, "Y", "nope")),
    ("fewer rows than parameters", lambda: ols(frame.head(2), "Y", "X")),
):
    try:
        bad()
    except (ValueError, KeyError) as e:
        refused.append([label, type(e).__name__, str(e)[:90]])
table(refused, headers=("call", "raised", "why"))

## What this bought you

Four estimators that each know which verdict licenses them, checked against worlds whose
truth is known — and a picture of what the licensed route buys, in the units of the effect
itself, rather than an argument about methodology.